# FIREWATCH observation-volume forecast

This notebook inspects the normalized FIRMS archive, builds a daily source-row count series, runs the seasonal-window forecast, and reviews its chronological holdout against a one-year seasonal-naive baseline.

**Target:** valid FIRMS observation rows per UTC day. This is not fire occurrence, spread, severity, or risk. A zero-row day is a zero in the row-count target, not proof of no fire or sensor coverage. The current archive is a historical snapshot, not a live alert feed. No GPU is needed for this lightweight statistical method.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / 'src' / 'firewatch' / 'forecast.py').is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError('Open this notebook from the FIREWATCH project folder.')
sys.path.insert(0, str(project_root))

from src.firewatch.api import DEFAULT_DATA_PATH, OBSERVATION_COLUMNS
from src.firewatch.forecast import forecast_observation_volume

print('Project:', project_root)
print('Dataset:', DEFAULT_DATA_PATH)

In [ ]:
observations = pd.read_parquet(DEFAULT_DATA_PATH, columns=OBSERVATION_COLUMNS)
observations['observation_datetime_utc'] = pd.to_datetime(
    observations['observation_datetime_utc'], utc=True
)
display(observations.head())
print(f'{len(observations):,} rows | {observations.observation_datetime_utc.min()} to {observations.observation_datetime_utc.max()}')

## Select the platforms and processing streams

Keep both settings as `None` for the all-source default, or list values such as `['Aqua', 'N20']` and `['archive']`. Results depend on these filters because each sensor and processing stream contributes different row volumes.

In [ ]:
satellites = None  # Example: ['Aqua', 'N20']
streams = None    # Example: ['archive']

scope = observations.loc[observations.validation_status.eq('valid')].copy()
if satellites:
    scope = scope.loc[scope.satellite.isin(satellites)]
if streams:
    scope = scope.loc[scope.stream_type.isin(streams)]

display(scope.groupby(['satellite', 'product_code', 'stream_type'], as_index=False)
              .agg(rows=('observation_id', 'size'))
              .sort_values(['satellite', 'stream_type']))
print(f'Selected valid rows: {len(scope):,}')

## Build the daily count series

Calendar dates without records are filled with zero because the forecast target is the number of rows in the extract. That does not convert the date into a verified no-fire or clear-coverage observation.

In [ ]:
daily_rows = (
    scope.set_index('observation_datetime_utc')['observation_id']
         .resample('D').size()
)
display(daily_rows.rename('observation_rows').describe().to_frame())
print('Calendar days:', len(daily_rows), '| zero-row days:', int(daily_rows.eq(0).sum()))

In [ ]:
plt.figure(figsize=(14, 4))
daily_rows.plot(color='#536d59', linewidth=1)
plt.title('Daily FIRMS source-row volume')
plt.ylabel('Valid observation rows')
plt.xlabel('UTC date')
plt.grid(axis='y', alpha=.2)
plt.tight_layout()
plt.show()

## Run and evaluate the forecast

The method averages a 15-day window centered on the matching date one year earlier. Backtesting predicts later archive dates using only their previous-year window and compares against the count from exactly 365 days earlier. The 90th-percentile watch is an in-app row-volume review threshold, not a fire alarm.

In [ ]:
result = forecast_observation_volume(scope, horizon_days=7)
display(result)

In [ ]:
if result['status'] == 'ok':
    backtest = result['backtest']
    score_table = pd.DataFrame([
        {'method': 'Seasonal 15-day window', **backtest['model']},
        {'method': 'Same date, 365 days earlier', **backtest['seasonal_naive_365d']},
    ])
    display(score_table)
    print(f"Rolling holdout days: {backtest['holdout_days']:,}")
    print(f"Model lower MAE than baseline: {backtest['beats_seasonal_naive_on_mae']}")
else:
    print(result)

In [ ]:
if result['status'] == 'ok':
    forecast_frame = pd.DataFrame(result['forecasts'])
    view = daily_rows.tail(90)
    plt.figure(figsize=(14, 5))
    view.plot(label='Observed archive rows', color='#536d59', linewidth=1.2)
    future_dates = pd.to_datetime(forecast_frame['date'], utc=True)
    plt.plot(future_dates, forecast_frame['expected_rows'], marker='o', color='#c06d46', label='Seasonal forecast')
    plt.axvline(pd.to_datetime(result['data_as_of'], utc=True), color='#8b9289', linestyle='--', label='Archive cutoff')
    plt.title('Archive tail and seven-day row-volume forecast')
    plt.ylabel('Valid observation rows')
    plt.xlabel('UTC date')
    plt.legend()
    plt.grid(axis='y', alpha=.2)
    plt.tight_layout()
    plt.show()
    display(forecast_frame)

## Interpretation

Review the MAE and WAPE against the baseline before trusting the projection. FIREWATCH currently has only about two years of data and the latest snapshot may be stale. This model predicts extract row volume only; it is not evidence of future fires, fire danger, or sensor coverage. The in-app alerts are archive review cues and no email, SMS, or browser push is sent.